# Preprocessing of Smart Card data from Santiago, Chile for analysis and ML model implementation

Last reviewed: Monday 06-04-2026

Times this file has been edited: 1

Hardware specs:


1. *Portable* branch:
    * Machine: MacBook Air (13-inch, 2017)
    * OS: MacOS Monterey v12.7.6
    * CPU: 1.8 GHz Intel Core i5 de dos núcleos
    * RAM: 8 GB 1600 MHz DDR3
    * Graphics: Intel HD Graphics 6000 1536 MB

2. *House* branch:
    * OS: Windows 10 Home 64-bit (10.0, Build 19045)
    * Processor: Intel(R) Core(TM) i7-8700K CPU @ 3.70GHz (12 CPUs), ~3.7GHz
    * Memory: 12288MB RAM
    * GPU: NVIDIA GeForce RTX 3060 12115 MB

In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import duckdb
from pathlib import Path

## Script 01: Conversión de datos a formato Parquet

In [6]:
"""
Script 01: Conversión de datos raw a formato Parquet
Sin transformaciones — solo lectura y escritura eficiente.
Datasets: Trips, Stages, GPS
"""

# ──── Configuración de DuckDB para manejo eficiente de grandes archivos ──────

def crear_conexion() -> duckdb.DuckDBPyConnection:
    """
    Crea una conexión DuckDB con límites de recursos seguros.
    Ajusta memory_limit según tu RAM disponible:
      - 8 GB RAM  → usa '4GB'
      - 16 GB RAM → usa '8GB'
      - 32 GB RAM → usa '16GB'
    """
    con = duckdb.connect()
    con.execute("SET memory_limit = '6GB'")       # ajusta según tu RAM
    con.execute("SET threads = 6")                 # ajusta según tus núcleos
    con.execute("SET temp_directory = 'D:/temp_duckdb'")  # disco para overflow
    return con

# ── Rutas base ───────────────────────────────────────────────────────────────
BASE_DIR  = Path(r"D:\GitHub\tesis_magister_route_choice_modelling\data")
RAW_DIR   = BASE_DIR / "raw"
PROC_DIR  = BASE_DIR / "parquets"


# ── Funciones auxiliares ─────────────────────────────────────────────────────

def asegurar_carpeta(path: Path):
    path.mkdir(parents=True, exist_ok=True)


def archivos_de_carpeta(carpeta: Path, extensiones: list[str]) -> list[Path]:
    archivos = []
    for ext in extensiones:
        archivos.extend(sorted(carpeta.glob(f"*{ext}")))
    return archivos


def reportar_compresion(archivos_raw: list[Path], parquet_path: Path):
    size_raw     = sum(f.stat().st_size for f in archivos_raw) / 1e6
    size_parquet = parquet_path.stat().st_size / 1e6
    ratio        = size_raw / size_parquet if size_parquet > 0 else 0
    print(f"  Raw:     {size_raw:.1f} MB")
    print(f"  Parquet: {size_parquet:.1f} MB")
    print(f"  Ratio:   {ratio:.1f}x")


# ── Trips ────────────────────────────────────────────────────────────────────

def convertir_trips(anio: str):
    carpeta_in  = RAW_DIR  / "Trips" / anio
    carpeta_out = PROC_DIR / "Trips" / anio
    asegurar_carpeta(carpeta_out)

    archivos = archivos_de_carpeta(carpeta_in, [".csv", ".viajes"])
    if not archivos:
        print(f"  [AVISO] No se encontraron archivos en {carpeta_in}")
        return

    salida = carpeta_out / f"trips_{anio}_raw.parquet"
    print(f"\n→ Trips {anio}: {len(archivos)} archivo(s)")

    # Construimos la query como UNION ALL de todos los archivos
    # DuckDB escribe directo a Parquet sin pasar por memoria Python
    union_parts = []
    for archivo in archivos:
        fecha = archivo.name.split(".")[0]
        # Usamos barras normales — DuckDB en Windows acepta ambas
        ruta = str(archivo).replace("\\", "/")
        union_parts.append(f"""
            SELECT
                '{fecha}' AS fecha,
                *
            FROM read_csv(
                '{ruta}',
                delim         = '|',
                header        = true,
                encoding      = 'cp1252',
                all_varchar   = true,
                null_padding  = true,
                ignore_errors = true
            )
        """)

    query_union = "\nUNION ALL\n".join(union_parts)
    salida_str  = str(salida).replace("\\", "/")

    con = crear_conexion()
    con.execute(f"""
        COPY (
            {query_union}
        )
        TO '{salida_str}'
        (FORMAT PARQUET, COMPRESSION 'ZSTD', ROW_GROUP_SIZE 100000)
    """)

    # Contar filas sin cargar en memoria
    n_filas = con.execute(f"""
        SELECT COUNT(*) FROM read_parquet('{salida_str}')
    """).fetchone()[0]
    con.close()

    print(f"  ✓ {salida.name} — {n_filas:,} filas totales")
    reportar_compresion(archivos, salida)


# ── Stages ───────────────────────────────────────────────────────────────────

def convertir_stages(anio: str):
    carpeta_in  = RAW_DIR  / "Stages" / anio
    carpeta_out = PROC_DIR / "Stages" / anio
    asegurar_carpeta(carpeta_out)

    archivos = archivos_de_carpeta(carpeta_in, [".csv", ".etapas"])
    if not archivos:
        print(f"  [AVISO] No se encontraron archivos en {carpeta_in}")
        return

    salida = carpeta_out / f"stages_{anio}_raw.parquet"
    print(f"\n→ Stages {anio}: {len(archivos)} archivo(s)")

    union_parts = []
    for archivo in archivos:
        fecha = archivo.name.split(".")[0]
        ruta  = str(archivo).replace("\\", "/")
        union_parts.append(f"""
            SELECT
                '{fecha}' AS fecha,
                *
            FROM read_csv(
                '{ruta}',
                delim         = '|',
                header        = true,
                encoding      = 'cp1252',
                all_varchar   = true,
                null_padding  = true,
                ignore_errors = true
            )
        """)

    query_union = "\nUNION ALL\n".join(union_parts)
    salida_str  = str(salida).replace("\\", "/")

    con = crear_conexion()
    con.execute(f"""
        COPY (
            {query_union}
        )
        TO '{salida_str}'
        (FORMAT PARQUET, COMPRESSION 'ZSTD', ROW_GROUP_SIZE 100000)
    """)

    n_filas = con.execute(f"""
        SELECT COUNT(*) FROM read_parquet('{salida_str}')
    """).fetchone()[0]
    con.close()

    print(f"  ✓ {salida.name} — {n_filas:,} filas totales")
    reportar_compresion(archivos, salida)


# ── GPS ──────────────────────────────────────────────────────────────────────

def convertir_gps(anio: str):
    carpeta_in  = RAW_DIR  / "GPS" / anio
    carpeta_out = PROC_DIR / "GPS" / anio
    asegurar_carpeta(carpeta_out)

    archivos = archivos_de_carpeta(carpeta_in, [".gps", ".csv"])
    if not archivos:
        print(f"  [AVISO] No se encontraron archivos en {carpeta_in}")
        return

    salida = carpeta_out / f"gps_{anio}_raw.parquet"
    print(f"\n→ GPS {anio}: {len(archivos)} archivo(s)")

    union_parts = []
    for archivo in archivos:
        fecha = archivo.name.split(".")[0]
        ruta  = str(archivo).replace("\\", "/")
        union_parts.append(f"""
            SELECT
                '{fecha}' AS fecha_archivo,
                *
            FROM read_csv(
                '{ruta}',
                delim         = ';',
                header        = false,
                encoding      = 'cp1252',
                all_varchar   = true,
                ignore_errors = true
            )
        """)

    query_union = "\nUNION ALL\n".join(union_parts)
    salida_str  = str(salida).replace("\\", "/")

    con = crear_conexion()
    con.execute(f"""
        COPY (
            {query_union}
        )
        TO '{salida_str}'
        (FORMAT PARQUET, COMPRESSION 'ZSTD', ROW_GROUP_SIZE 100000)
    """)

    n_filas = con.execute(f"""
        SELECT COUNT(*) FROM read_parquet('{salida_str}')
    """).fetchone()[0]
    con.close()

    print(f"  ✓ {salida.name} — {n_filas:,} filas totales")
    reportar_compresion(archivos, salida)

In [7]:
# ── Ejecución ────────────────────────────────────────────────────────────────

if __name__ == "__main__":

    print("=" * 60)
    print("CONVERSIÓN RAW → PARQUET (sin transformaciones)")
    print("=" * 60)

    for anio in ["2024", "2025"]: # Agregar otros años según disponibilidad y necesidad
        convertir_trips(anio)

    for anio in ["2025"]:
        convertir_stages(anio)

    for anio in ["2018", "2022"]:
        convertir_gps(anio)

    print("\n" + "=" * 60)
    print("✓ Conversión completada")
    print("=" * 60)

CONVERSIÓN RAW → PARQUET (sin transformaciones)

→ Trips 2024: 9 archivo(s)
  ✓ trips_2024_raw.parquet — 24,430,419 filas totales
  Raw:     9974.8 MB
  Parquet: 1727.0 MB
  Ratio:   5.8x

→ Trips 2025: 7 archivo(s)
  ✓ trips_2025_raw.parquet — 21,313,043 filas totales
  Raw:     8778.0 MB
  Parquet: 1535.9 MB
  Ratio:   5.7x

→ Stages 2025: 7 archivo(s)
  ✓ stages_2025_raw.parquet — 27,949,622 filas totales
  Raw:     8249.0 MB
  Parquet: 1340.6 MB
  Ratio:   6.2x

→ GPS 2018: 7 archivo(s)
  ✓ gps_2018_raw.parquet — 77,761,905 filas totales
  Raw:     6008.3 MB
  Parquet: 1172.1 MB
  Ratio:   5.1x

→ GPS 2022: 14 archivo(s)
  ✓ gps_2022_raw.parquet — 172,111,519 filas totales
  Raw:     12123.8 MB
  Parquet: 1304.5 MB
  Ratio:   9.3x

✓ Conversión completada


## Script 02: Agregación del clima con API externa

In [9]:
"""
Script 02: Descarga de datos climáticos históricos desde Open-Meteo
para los períodos cubiertos por los datasets de viajes y etapas.

Estrategia:
- Se descarga una vez por período y se guarda como Parquet independiente.
- El join con Trips/Stages se realiza en tiempo de consulta con DuckDB.
- No se duplican datos climáticos en cada dataset.
- Resolución: horaria, coordenadas centradas en Santiago.
"""

import openmeteo_requests
import requests_cache
import pandas as pd
from retry_requests import retry
from pathlib import Path

# ── Rutas base ───────────────────────────────────────────────────────────────
BASE_DIR  = Path(r"D:\GitHub\tesis_magister_route_choice_modelling\data")
PROC_DIR  = BASE_DIR / "parquets"
CLIMA_DIR = PROC_DIR / "02_enriched" / "clima"


# ── Coordenadas de Santiago ──────────────────────────────────────────────────
# Centro geográfico aproximado del Gran Santiago
# Suficientemente preciso para clima — la variación intra-ciudad es mínima
# para temperatura y precipitación a escala horaria
SANTIAGO_LAT =  -33.4489
SANTIAGO_LON =  -70.6693
TIMEZONE     = "America/Santiago"


# ── Variables a descargar ────────────────────────────────────────────────────
VARIABLES_HORARIAS = [
    "temperature_2m",       # temperatura en °C
    "precipitation",        # precipitación total mm/h
    "rain",                 # lluvia (excluye nieve) mm/h
    "weather_code",         # código WMO de condición meteorológica
    "wind_speed_10m",       # velocidad del viento km/h
    "relative_humidity_2m", # humedad relativa %
]


# ── Períodos a descargar ─────────────────────────────────────────────────────
# Agregar aquí todos los períodos que tengas en tus datasets
# formato: (nombre_periodo, fecha_inicio, fecha_fin)
PERIODOS = [
    ("2019_mayo",      "2019-05-19", "2019-05-19"),
    ("2024_noviembre", "2024-11-09", "2024-11-17"),
    ("2025_abril",     "2025-04-21", "2025-04-27"),
]


# ── Cliente Open-Meteo con caché y reintentos ────────────────────────────────
def crear_cliente():
    """
    Crea un cliente Open-Meteo con:
    - Caché en disco: evita re-descargar si el script se ejecuta más de una vez
    - Reintentos automáticos: maneja cortes de red transitorios
    """
    cache_session = requests_cache.CachedSession(
        str(BASE_DIR / "processed" / "02_enriched" / ".cache_openmeteo"),
        expire_after = -1  # caché permanente — datos históricos no cambian
    )
    retry_session = retry(cache_session, retries=5, backoff_factor=0.2)
    return openmeteo_requests.Client(session=retry_session)


# ── Descarga y procesamiento ─────────────────────────────────────────────────

def descargar_clima_periodo(
    cliente,
    nombre: str,
    fecha_inicio: str,
    fecha_fin: str
) -> pd.DataFrame:
    """
    Descarga datos horarios de Open-Meteo para un período dado
    y devuelve un DataFrame con una fila por hora.
    """
    print(f"  Descargando {nombre} ({fecha_inicio} → {fecha_fin})...")

    params = {
        "latitude":   SANTIAGO_LAT,
        "longitude":  SANTIAGO_LON,
        "start_date": fecha_inicio,
        "end_date":   fecha_fin,
        "hourly":     VARIABLES_HORARIAS,
        "timezone":   TIMEZONE,
    }

    respuestas = cliente.weather_api(
        "https://archive-api.open-meteo.com/v1/archive",
        params=params
    )
    resp = respuestas[0]  # una sola ubicación → primer elemento

    # Extraer datos horarios
    horario = resp.Hourly()

    df = pd.DataFrame({
        "timestamp": pd.date_range(
            start = pd.Timestamp(horario.Time(), unit="s", tz=TIMEZONE),
            end   = pd.Timestamp(horario.TimeEnd(), unit="s", tz=TIMEZONE),
            freq  = pd.Timedelta(seconds=horario.Interval()),
            inclusive = "left"
        ),
        "temperature_2m":       horario.Variables(0).ValuesAsNumpy(),
        "precipitation":        horario.Variables(1).ValuesAsNumpy(),
        "rain":                 horario.Variables(2).ValuesAsNumpy(),
        "weather_code":         horario.Variables(3).ValuesAsNumpy(),
        "wind_speed_10m":       horario.Variables(4).ValuesAsNumpy(),
        "relative_humidity_2m": horario.Variables(5).ValuesAsNumpy(),
    })

    # Columnas derivadas útiles para el modelo
    df["fecha"]         = df["timestamp"].dt.date.astype(str)
    df["hora"]          = df["timestamp"].dt.hour
    df["mediahora"]     = (df["hora"] * 2 + (df["timestamp"].dt.minute >= 30).astype(int))
    df["llueve"]        = (df["rain"] > 0.1).astype(int)          # binario: llueve o no
    df["lluvia_intensa"]= (df["rain"] > 2.0).astype(int)          # > 2mm/h = lluvia intensa
    df["periodo_clima"] = nombre

    # Clasificación de condición meteorológica según código WMO
    # https://open-meteo.com/en/docs — sección Weather variable descriptions
    df["condicion"] = df["weather_code"].map(clasificar_wmo).fillna("desconocido")

    return df


def clasificar_wmo(codigo: float) -> str:
    """
    Clasifica el código WMO en categorías legibles.
    Referencia: https://open-meteo.com/en/docs (WMO Weather interpretation codes)
    """
    if pd.isna(codigo):
        return "desconocido"
    c = int(codigo)
    if c == 0:                    return "despejado"
    elif c in (1, 2, 3):          return "nublado"
    elif c in (45, 48):           return "niebla"
    elif c in (51, 53, 55):       return "llovizna"
    elif c in (61, 63, 65):       return "lluvia"
    elif c in (71, 73, 75, 77):   return "nieve"
    elif c in (80, 81, 82):       return "chubascos"
    elif c in (95, 96, 99):       return "tormenta"
    else:                         return "otro"


# ── Ejecución ────────────────────────────────────────────────────────────────

def asegurar_carpeta(path: Path):
    path.mkdir(parents=True, exist_ok=True)


if __name__ == "__main__":

    asegurar_carpeta(CLIMA_DIR)

    print("=" * 60)
    print("DESCARGA DE DATOS CLIMÁTICOS — Open-Meteo")
    print(f"Ubicación: Santiago ({SANTIAGO_LAT}, {SANTIAGO_LON})")
    print("=" * 60)

    cliente = crear_cliente()
    dfs = []

    for nombre, fecha_inicio, fecha_fin in PERIODOS:
        salida = CLIMA_DIR / f"clima_{nombre}.parquet"

        # Si ya existe el archivo, no re-descarga
        if salida.exists():
            print(f"  [SKIP] {salida.name} ya existe — usa caché o borra para re-descargar")
            continue

        df = descargar_clima_periodo(cliente, nombre, fecha_inicio, fecha_fin)
        df.to_parquet(salida, compression="zstd", index=False)

        print(f"  ✓ {salida.name} — {len(df):,} filas ({df['llueve'].sum()} horas con lluvia)")
        dfs.append(df)

    # Unificar todos los períodos en un solo archivo de referencia
    salida_total = CLIMA_DIR / "clima_todos_periodos.parquet"
    if not salida_total.exists():
        todos = pd.concat(
            [pd.read_parquet(f) for f in sorted(CLIMA_DIR.glob("clima_*.parquet"))
             if "todos" not in f.name],
            ignore_index=True
        )
        todos.to_parquet(salida_total, compression="zstd", index=False)
        print(f"\n  ✓ Archivo unificado: {salida_total.name} — {len(todos):,} filas totales")

    print("\n" + "=" * 60)
    print("✓ Descarga completada")
    print("=" * 60)

DESCARGA DE DATOS CLIMÁTICOS — Open-Meteo
Ubicación: Santiago (-33.4489, -70.6693)
  Descargando 2019_mayo (2019-05-19 → 2019-05-19)...
  ✓ clima_2019_mayo.parquet — 24 filas (0 horas con lluvia)
  Descargando 2024_noviembre (2024-11-09 → 2024-11-17)...
  ✓ clima_2024_noviembre.parquet — 216 filas (1 horas con lluvia)
  Descargando 2025_abril (2025-04-21 → 2025-04-27)...
  ✓ clima_2025_abril.parquet — 168 filas (0 horas con lluvia)

  ✓ Archivo unificado: clima_todos_periodos.parquet — 408 filas totales

✓ Descarga completada


## Script 03: Agregación de rutas y horarios mediante GTFS del sistema RED (vigente desde abril 2026)